# NVS Benchmark - Colab Full Run

This notebook runs all methods (nerf_static, nerf_dynamic, gs_static, gs_dynamic) on all scenes from Blender Synthetic and D-NeRF, using preset standard.

Notes:
- The full run can take many hours and may exceed Colab limits.
- Keep Drive mounted to persist data, logs, and artifacts.
- The report uses a summary snapshot averaged over all scenes.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount failed or not running in Colab:', exc)

PROJECT_ROOT = Path('/content/TCC-Source-Code')
DRIVE_ROOT = Path('/content/drive/MyDrive/nvs_benchmark')
DATA_DIR = DRIVE_ROOT / 'data'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'
LOGS_DIR = DRIVE_ROOT / 'logs'
CACHE_DIR = DRIVE_ROOT / 'cache'

for path in [DATA_DIR, ARTIFACTS_DIR, LOGS_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['NVS_BENCHMARK_CACHE_DIR'] = str(CACHE_DIR)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DRIVE_ROOT:', DRIVE_ROOT)

In [ ]:
import subprocess

REPO_URL = 'https://github.com/PedroHeinrichSP/TCC-Source-Code.git'

if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print('Repo already present:', PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print('CWD:', os.getcwd())

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

os.environ['PYTHONPATH'] = str(PROJECT_ROOT / 'src')

In [ ]:
def run_cmd(cmd):
    print('Running:', ' '.join(cmd))
    return subprocess.run(cmd, check=False)

run_cmd([sys.executable, '-m', 'nvs_benchmark.cli', 'install', '--catalog-file', './configs/install_catalog.json', '--only', 'datasets', '--execute'])

In [ ]:
run_cmd([sys.executable, '-m', 'nvs_benchmark.cli', 'status'])
run_cmd([sys.executable, '-m', 'nvs_benchmark.cli', 'list-datasets', '--check-files'])
run_cmd([sys.executable, '-m', 'nvs_benchmark.cli', 'list-methods'])

In [ ]:
import json
import time

METHODS = ['nerf_static', 'nerf_dynamic', 'gs_static', 'gs_dynamic']

DATASETS = {
    'blender_synthetic': {
        'root': DATA_DIR / 'blender_synthetic' / 'nerf_synthetic',
        'scenes': ['lego', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship'],
    },
    'd_nerf': {
        'root': DATA_DIR / 'd_nerf',
        'scenes': ['bouncing_balls', 'hell_warrior', 'hook', 'jumping_jacks', 'lego', 'mutant', 'stand_up', 't_rex'],
    },
}

TMP_SNAPSHOT = ARTIFACTS_DIR / 'metrics' / 'tmp_snapshot.json'
RESULTS_FILE = ARTIFACTS_DIR / 'metrics' / 'benchmark_matrix.json'
SUMMARY_SNAPSHOT = ARTIFACTS_DIR / 'metrics' / 'benchmark_all_summary.json'

RUN_RESULTS = []

def load_snapshot(path):
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def append_result(payload):
    RUN_RESULTS.append(payload)
    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with RESULTS_FILE.open('w', encoding='utf-8') as handle:
        json.dump(RUN_RESULTS, handle, indent=2)

for dataset, spec in DATASETS.items():
    for scene in spec['scenes']:
        scene_root = spec['root'] / scene
        for method in METHODS:
            print('=' * 80)
            print('Dataset:', dataset, 'Scene:', scene, 'Method:', method)
            print('=' * 80)
            cmd = [
                sys.executable, '-m', 'nvs_benchmark.cli', 'method-run',
                '--method', method,
                '--dataset', dataset,
                '--root', str(scene_root),
                '--split', 'train',
                '--output-dir', str(ARTIFACTS_DIR),
                '--log-dir', str(LOGS_DIR),
                '--compute-metrics',
                '--preset', 'standard',
                '--snapshot-file', str(TMP_SNAPSHOT),
                '--reuse-renders',
                '--reuse-metrics',
                '--adaptive-preset',
            ]

            started = time.time()
            result = subprocess.run(cmd, check=False)
            elapsed = time.time() - started

            snapshot = load_snapshot(TMP_SNAPSHOT) or {}
            metrics = snapshot.get(method) if isinstance(snapshot, dict) else None
            status = 'ok' if result.returncode == 0 and metrics else 'failed'

            append_result({
                'dataset': dataset,
                'scene': scene,
                'method': method,
                'status': status,
                'returncode': result.returncode,
                'elapsed_seconds': round(elapsed, 2),
                'metrics': metrics,
            })

print('Run complete. Results written to:', RESULTS_FILE)

In [ ]:
from collections import defaultdict

def safe_mean(values):
    filtered = [v for v in values if isinstance(v, (int, float))]
    if not filtered:
        return None
    return sum(filtered) / len(filtered)

by_method = defaultdict(list)
for entry in RUN_RESULTS:
    if entry.get('status') != 'ok':
        continue
    metrics = entry.get('metrics') or {}
    by_method[entry['method']].append(metrics)

summary = {}
for method, items in by_method.items():
    summary[method] = {
        'method': method,
        'pairs': safe_mean([m.get('pairs') for m in items]),
        'psnr': safe_mean([m.get('psnr') for m in items]),
        'ssim': safe_mean([m.get('ssim') for m in items]),
        'lpips': safe_mean([m.get('lpips') for m in items]),
        'fps': safe_mean([m.get('fps') for m in items]),
        'vram_gb': safe_mean([m.get('vram_gb') for m in items]),
        'train_seconds': safe_mean([m.get('train_seconds') for m in items]),
        'inference_seconds': safe_mean([m.get('inference_seconds') for m in items]),
        'frame_time_ms': safe_mean([m.get('frame_time_ms') for m in items]),
        'latency_p50_ms': safe_mean([m.get('latency_p50_ms') for m in items]),
        'latency_p90_ms': safe_mean([m.get('latency_p90_ms') for m in items]),
        'latency_p99_ms': safe_mean([m.get('latency_p99_ms') for m in items]),
    }

SUMMARY_SNAPSHOT.parent.mkdir(parents=True, exist_ok=True)
with SUMMARY_SNAPSHOT.open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)

print('Summary snapshot written to:', SUMMARY_SNAPSHOT)

In [ ]:
run_cmd([
    sys.executable, '-m', 'nvs_benchmark.cli', 'report-generate',
    '--snapshot-file', str(SUMMARY_SNAPSHOT),
    '--output-dir', str(ARTIFACTS_DIR / 'reports'),
    '--report-name', 'benchmark_report',
    '--log-dir', str(LOGS_DIR),
    '--strict-snapshot',
    '--min-methods', '1',
    '--require-finite-metrics',
])

In [ ]:
import pandas as pd

baseline_path = PROJECT_ROOT / 'configs' / 'paper_baselines.json'
with baseline_path.open('r', encoding='utf-8') as handle:
    baselines = json.load(handle)

summary_data = {}
if SUMMARY_SNAPSHOT.exists():
    with SUMMARY_SNAPSHOT.open('r', encoding='utf-8') as handle:
        summary_data = json.load(handle)

rows = []
for method, baseline in baselines.get('baselines', {}).items():
    avg = baseline.get('average', {})
    bench = summary_data.get(method, {})
    rows.append({
        'method': method,
        'paper_psnr': avg.get('psnr'),
        'paper_ssim': avg.get('ssim'),
        'paper_lpips': avg.get('lpips'),
        'bench_psnr': bench.get('psnr'),
        'bench_ssim': bench.get('ssim'),
        'bench_lpips': bench.get('lpips'),
    })

df = pd.DataFrame(rows)
df['delta_psnr'] = df['bench_psnr'] - df['paper_psnr']
df['delta_ssim'] = df['bench_ssim'] - df['paper_ssim']
df['delta_lpips'] = df['bench_lpips'] - df['paper_lpips']

print(df)

In [ ]:
print('Artifacts root:', ARTIFACTS_DIR)
print('Metrics files:', list((ARTIFACTS_DIR / 'metrics').glob('*.json')))
print('Reports:', list((ARTIFACTS_DIR / 'reports').glob('*')))